# Tools in AutoGen [Agent Patterns - Module 13]

> **MLCourse - Agentic AI - Agent Patterns**

Every framework in this course wraps tool calling. What differs is how much
ceremony sits between your Python function and the model.

AutoGen's answer is unusually light: **pass the function**. It reads the
signature and the docstring and builds the schema itself.

### What you will learn

1. Registering a plain Python function as a tool.
2. The tool-call message types in the transcript.
3. `reflect_on_tool_use` - returning the raw result vs summarising it.
4. Why tool schemas come from your type hints and docstrings.

### Key takeaways

- Type hints and docstrings ARE the tool schema. Write them properly.
- The transcript shows the call and the result as separate messages.
- Tool results are untrusted input; the injection rules still apply.

### Setup: imports, environment, track discovery


In [ ]:
import os
import sys
import json
import time
import random
import asyncio
from pathlib import Path
from dotenv import load_dotenv

def _find_track(depth=6):
    p = Path.cwd()
    for _ in range(depth):
        if (p / "03_agentic_ai").is_dir():
            return p
        p = p.parent
    return Path.cwd()

TRACK = _find_track()
# The .env lives INSIDE the track folder, not the repo root that
# _find_track() returns - joining ".env" onto TRACK alone is a silent no-op.
load_dotenv(TRACK / "03_agentic_ai" / ".env", override=False)

GROQ_API_KEY = os.environ["GROQ_API_KEY"]   # loud failure if missing, by design
MODEL = "qwen/qwen3.8-27b"                   # Groq-hosted; never OpenAI

print(f"Track root : {TRACK}")
print(f"Model      : {MODEL} (via Groq's OpenAI-compatible endpoint)")


### Point AutoGen at Groq


In [ ]:
# AutoGen ships an OpenAI client. Groq exposes an OpenAI-COMPATIBLE endpoint,
# so we reuse that client and only change the base_url. This is the standard
# way to run AutoGen on a non-OpenAI provider - there is no Groq-specific
# client to install.
#
# `model_info` is REQUIRED for any model AutoGen does not have a built-in
# capability table for. Without it you get a ValueError before a single
# request goes out. You are telling the framework what the model can do.

from autogen_ext.models.openai import OpenAIChatCompletionClient

def make_client():
    return OpenAIChatCompletionClient(
        model=MODEL,
        api_key=GROQ_API_KEY,
        base_url="https://api.groq.com/openai/v1",   # <- the only Groq-specific line
        temperature=0.0,
        max_tokens=500,                              # free tier is 8000 TPM
        model_info={
            "vision": False,
            "function_calling": True,
            "json_output": True,
            "family": "unknown",
            "structured_output": False,
        },
    )

print("make_client() ready")


### 1. A function is a tool

No decorator, no `Tool` subclass, no JSON schema. AutoGen introspects:

- the **function name** becomes the tool name,
- the **docstring** becomes the description the model reads,
- the **type hints** become the parameter schema.

Which means your docstring is now prompt engineering. A vague docstring is a
vague tool description is a model that calls the wrong tool.

### Two small, deterministic tools


In [ ]:
INVENTORY = {
    "NW-1001": {"name": "Brass elbow 15mm", "stock": 184, "price": 3.40},
    "NW-1002": {"name": "Copper tee 22mm", "stock": 0, "price": 5.95},
    "NW-2100": {"name": "Compression valve 15mm", "stock": 6, "price": 11.25},
}

def check_stock(sku: str) -> str:
    """Look up the current stock level and price for a product SKU.

    Args:
        sku: the product code, formatted like NW-1001.
    """
    item = INVENTORY.get(sku.upper().strip())
    if item is None:
        return f"No product found with SKU {sku}."
    return (f"{sku}: {item['name']}, {item['stock']} in stock, "
            f"{item['price']:.2f} each.")

def order_value(unit_price: float, quantity: int) -> str:
    """Calculate the total value of an order line.

    Args:
        unit_price: price of one unit.
        quantity: number of units.
    """
    return f"{quantity} x {unit_price:.2f} = {unit_price * quantity:.2f}"

print(check_stock("NW-2100"))
print(order_value(11.25, 4))


### Give them to an agent


In [ ]:
from autogen_agentchat.agents import AssistantAgent

client = make_client()

clerk = AssistantAgent(
    name="clerk",
    model_client=client,
    tools=[check_stock, order_value],       # plain functions - that is all
    system_message=("You are a parts desk clerk. Use the tools for any stock "
                    "or price question - never answer from memory. "
                    "Answer in one short sentence."),
    reflect_on_tool_use=True,
)

print("tools registered:", [t.name for t in clerk._tools])


### Run a query that needs both tools


In [ ]:
result = await clerk.run(
    task="How many NW-2100 do we have, and what would 4 of them cost?"
)

print(result.messages[-1].content)


### 2. Reading the tool transcript

The interesting part is not the answer, it is the message sequence. AutoGen
records the model's request and the tool's return as separate, typed
messages, which makes tool-calling agents genuinely debuggable.

### Every message, with its type


In [ ]:
for i, m in enumerate(result.messages):
    kind = type(m).__name__
    content = getattr(m, "content", "")
    print(f"[{i}] {kind}  (source={getattr(m, 'source', '-')})")
    if isinstance(content, list):
        for c in content:
            print(f"      {c}")
    else:
        print(f"      {str(content)[:200]}")
    print()


You should see, in order:

1. `TextMessage` - the task,
2. `ToolCallRequestEvent` - the model asking for `check_stock(sku=...)`,
3. `ToolCallExecutionEvent` - what the function actually returned,
4. more of the same if a second tool was needed,
5. `TextMessage` - the final answer.

Compare that with debugging a chain where the tool call is a string inside
one blob of output. The typed transcript is AutoGen's real advantage.

### 3. `reflect_on_tool_use`

This flag decides what the agent does after a tool returns:

- **`True`** - feed the result back to the model and let it write a natural
  answer. Costs one extra LLM call. Good for user-facing replies.
- **`False`** - return the tool's raw output as the agent's response. Free,
  exact, and correct when the tool output is already the answer (a number, a
  status, a row of JSON).

Beginners leave it on `True` everywhere and pay for a model call to rephrase
`"6 in stock"` into `"There are 6 in stock."` Choose per agent.

### Compare the two modes


In [ ]:
raw_clerk = AssistantAgent(
    name="raw_clerk",
    model_client=client,
    tools=[check_stock],
    system_message="Use the tool to answer stock questions.",
    reflect_on_tool_use=False,          # <- raw tool output
)

r = await raw_clerk.run(task="Stock for NW-1002?")
print("reflect_on_tool_use=False ->", r.messages[-1].content)
print()
print("messages:", [type(m).__name__ for m in r.messages])
print()
print("With reflect=True the same query costs one MORE model call, to turn")
print("the tool string into a sentence. Worth it for a chat UI, wasteful for")
print("a pipeline step whose output feeds more code.")


### 4. Tool results are untrusted input

Everything from `05_production_security/01_prompt_injection` applies here
without modification. A tool that reads a file, queries a database of
user-submitted content, or fetches a web page returns **text an attacker may
have written**, and that text goes straight into the model's context.

The defenses are the ones you already know:

- constrain the tool's output shape rather than returning free text,
- validate arguments in the function before acting on them,
- never let a tool result decide whether another tool runs, if that tool does
  anything irreversible,
- keep the destructive tools out of the agent's hands entirely and put a
  human in front of them (`06_agent_patterns/14_async_human_approval`).

Note that our `check_stock` does the small right thing already: it looks the
SKU up in a dict and returns a fixed message. There is no path from the
model's argument to arbitrary behaviour.

### Argument validation in the tool itself


In [ ]:
# The model chooses the arguments. The FUNCTION decides what is acceptable.

import re as _re

def check_stock_strict(sku: str) -> str:
    """Look up stock for a product SKU formatted like NW-1001."""
    if not _re.fullmatch(r"NW-\d{4}", sku.upper().strip()):
        return f"Rejected: {sku!r} is not a valid SKU format."
    item = INVENTORY.get(sku.upper().strip())
    return f"{sku}: {item['stock']} in stock." if item else f"No product {sku}."

for attempt in ["NW-2100", "nw-1002", "'; DROP TABLE parts;--", "NW-99999"]:
    print(f"  {attempt!r:28s} -> {check_stock_strict(attempt)}")


### Clean up


In [ ]:
await client.close()
print("client closed")


### Pitfalls recap

- **Vague docstrings.** They are the tool description the model reads. A bad
  one causes wrong-tool calls that look like model stupidity.
- **Missing type hints.** No hints, no schema, no reliable calling.
- **`reflect_on_tool_use=True` by default everywhere.** It is an extra model
  call per tool use. Turn it off where the raw result is the answer.
- **Trusting tool output.** It is untrusted input the moment the tool touches
  anything a user can write to.
- **Unvalidated arguments.** The model picks them; your function must check
  them.

### Next

Notebook 04 puts AutoGen, LangGraph and CrewAI side by side on the same task.